# Gas Analyzer — Forecasting

Compare Prophet, XGBoost, and Ensemble forecasters on the CH4 time series.


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

from src.ingestion.loader import DataLoader
from src.processing.cleaner import DataCleaner
from src.forecasting.xgboost_model import XGBoostModel
from src.forecasting.sarima_model import SARIMAModel
from src.forecasting.ensemble import EnsembleForecaster
from src.visualization.plotter import GasPlotter

HORIZON = 48
TARGET = 'CH4'

data = DataLoader.load_sample()
cleaner = DataCleaner(normalization='none', resampling_freq='1h')
data_clean = cleaner.fit_transform(data)

series = data_clean[TARGET]
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]

plotter = GasPlotter('../outputs/plots')
print(f'Train: {len(train)} | Test: {len(test)}')

## XGBoost Forecast

In [ ]:
xgb = XGBoostModel(horizon=HORIZON, n_estimators=300, lookback=48)
xgb.fit(train)
xgb_result = xgb.predict(steps=HORIZON)

metrics = xgb.evaluate(test.values, xgb_result.predictions.values[:HORIZON])
print('XGBoost metrics:', metrics)

fig = plotter.plot_forecast(series, xgb_result, n_history=168)
plt.show()

## SARIMA Forecast

In [ ]:
sarima = SARIMAModel(horizon=HORIZON, order=(1,1,1), seasonal_order=(1,1,1,24))
sarima.fit(train)
sarima_result = sarima.predict(steps=HORIZON)

fig = plotter.plot_forecast(series, sarima_result, n_history=168)
plt.show()

## Ensemble (XGBoost + SARIMA)

In [ ]:
xgb2 = XGBoostModel(horizon=HORIZON, n_estimators=300, lookback=48)
sarima2 = SARIMAModel(horizon=HORIZON, order=(1,1,1), seasonal_order=(1,1,1,24))
ensemble = EnsembleForecaster([xgb2, sarima2], weights=[0.6, 0.4], horizon=HORIZON)
ensemble.fit(train)
ens_result = ensemble.predict(steps=HORIZON)

ens_metrics = xgb.evaluate(test.values, ens_result.predictions.values[:HORIZON])
print('Ensemble metrics:', ens_metrics)

fig = plotter.plot_forecast(series, ens_result, n_history=168, title='Ensemble Forecast: CH4')
plt.show()

## Cross-Validation

In [ ]:
xgb_cv = XGBoostModel(horizon=24, n_estimators=100, lookback=48)
cv_results = xgb_cv.cross_validate(series, n_splits=3, test_size=24)
display(cv_results)
print(f"\nMean RMSE: {cv_results['RMSE'].mean():.4f} +/- {cv_results['RMSE'].std():.4f}")